# Digital Twin rApp — Sionna RT Walkthrough

This notebook **re-runs the exact pipeline from the `dtrapp` codebase**, stage by
stage, on a scene you generated with the codebase, and **visualizes everything**
from the 3D world all the way to per-UE / per-cell **downlink throughput**.

It is structured like the official Sionna RT tutorials, but every *number*
(network layout, path gain, SINR, throughput) is produced by calling the
codebase's own functions — so the results are identical to running the CLI:

```
python3 -m dtrapp.runner.cli configs/example.yaml
```

**Pipeline:**

| Stage | What we do | Source of truth |
|------|------------|-----------------|
| 1. Geometry | load the codebase's `scene.xml`, reconstruct the scene extent | `dtrapp.geometry` (offline reuse) |
| 2. Network | regenerate the *same* cells + UEs | `dtrapp.network.RandomNetworkSource` |
| 3. Propagation | ray-trace path gain | `dtrapp.propagation.SionnaPropagationEngine` |
| 4-5. KPI | SINR -> Shannon throughput | `dtrapp.kpi.compute_kpis` |
| 6. Output | write the CSV/JSON | `dtrapp.runner.output.write_outputs` |

**What you need before running:** a scene folder produced by the codebase
(`output/scene/scene.xml` + `meshes/`) and the **same** config YAML (the `seed`
must match the run that made the scene).

## 0. Setup

Install the dependencies if needed, import everything, and point the notebook at
your repo + scene folder. **Edit the three paths in the next cell.**

In [ ]:
# Install dependencies if they are missing (uncomment to run once):
# %pip install sionna-rt numpy pyyaml matplotlib pandas

%matplotlib inline
import os
import sys
import glob
import math

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

# ----------------------- EDIT THESE THREE PATHS -----------------------
# Repo root = the folder that contains the `dtrapp/` package and `configs/`.
REPO_ROOT = os.path.abspath(os.environ.get("DTRAPP_REPO", ".."))
# The scene folder produced by the codebase (contains scene.xml + meshes/).
SCENE_DIR = os.path.join(REPO_ROOT, "output", "scene")
# The SAME config YAML used to generate that scene (the seed must match!).
CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "example.yaml")
# ----------------------------------------------------------------------

# Make the dtrapp package importable from the repo root.
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Set True if you are NOT in a notebook GUI: we then render images to file
# instead of opening the interactive 3D preview widget (which is Jupyter-only).
no_preview = False

print("REPO_ROOT :", REPO_ROOT)
print("SCENE_DIR :", SCENE_DIR)
print("CONFIG    :", CONFIG_PATH)
assert os.path.exists(os.path.join(SCENE_DIR, "scene.xml")), (
    "scene.xml not found - generate it first, e.g. "
    "`python3 -m dtrapp.runner.cli configs/example.yaml`."
)

## 1. Load the scenario config

We load the exact same `SimulationConfig` the codebase uses. Every downstream
stage reads its parameters from here.

In [ ]:
from dtrapp.config import SimulationConfig

config = SimulationConfig.from_yaml(CONFIG_PATH)

print("Loaded config:")
for k, v in config.to_dict().items():
    print(f"  {k}: {v}")

## 2. Stage 1 — load & visualize the 3D scene

We load the codebase's `scene.xml` into Sionna RT. We also **reconstruct the
scene extent** (the min/max x,y over all building footprints) directly from the
mesh files — this is exactly how `dtrapp.geometry.scene_builder` computes the
rectangle that the network generator uses to place base stations and UEs, so the
network we regenerate below will be identical.

In [ ]:
import sionna
from sionna.rt import load_scene, Camera

scene = load_scene(os.path.join(SCENE_DIR, "scene.xml"))
print("Scene loaded. Objects (merged by material):", len(scene.objects))


def read_ply(path):
    """Minimal ASCII-PLY reader for the meshes written by dtrapp."""
    with open(path, "r") as fh:
        lines = fh.read().splitlines()
    n_vert, header_end = 0, 0
    for i, ln in enumerate(lines):
        if ln.startswith("element vertex"):
            n_vert = int(ln.split()[-1])
        if ln.strip() == "end_header":
            header_end = i + 1
            break
    verts = [tuple(float(v) for v in ln.split()[:3])
             for ln in lines[header_end:header_end + n_vert]]
    return np.array(verts, dtype=float)


# Reconstruct the scene extent EXACTLY as the codebase does: min/max x,y over all
# building footprints (the ground plane is excluded). This is the rectangle the
# random generator samples base-station and UE positions from.
bldg_files = sorted(glob.glob(os.path.join(SCENE_DIR, "meshes", "bldg-*.ply")))
assert bldg_files, "No building meshes (bldg-*.ply) found in the scene folder."
all_xy = np.vstack([read_ply(p)[:, :2] for p in bldg_files])
extent_m = (
    float(all_xy[:, 0].min()), float(all_xy[:, 1].min()),
    float(all_xy[:, 0].max()), float(all_xy[:, 1].max()),
)
print(f"{len(bldg_files)} buildings")
print("extent_m (min_x, min_y, max_x, max_y) =",
      tuple(round(v, 2) for v in extent_m))

In [ ]:
# A handy aerial camera centered on the scene, reused for all renders.
cx = 0.5 * (extent_m[0] + extent_m[2])
cy = 0.5 * (extent_m[1] + extent_m[3])
span = max(extent_m[2] - extent_m[0], extent_m[3] - extent_m[1])
aerial_cam = Camera(position=[cx, cy - span, span], look_at=[cx, cy, 0.0])

# Interactive 3D viewer (in Jupyter) or a rendered still image (otherwise).
if no_preview:
    scene.render_to_file(camera=aerial_cam, filename="scene_overview.png",
                         resolution=[900, 600])
    display(Image("scene_overview.png"))
else:
    scene.preview()

## 3. Stage 2 — network data (cells + UEs)

We regenerate the network with the codebase's `RandomNetworkSource`, passing the
same `config` and the reconstructed `extent_m`. Because the generator is fully
seeded, this reproduces the **identical** base stations and UEs that a CLI run
would create.

> **Swapping in real data:** the only requirement is that a real source projects
> its BS/UE coordinates into the **same local metre frame** as the buildings
> (origin at the bbox centre). Implement `dtrapp.network.base.NetworkDataSource`
> and replace the line below — nothing downstream changes.

In [ ]:
from dtrapp.network import RandomNetworkSource

network = RandomNetworkSource(config, extent_m).generate()
print(f"{len(network.cells)} cells, {len(network.ues)} UEs\n")

print("First 3 cells:")
for c in network.cells[:3]:
    pos = tuple(round(p, 1) for p in c.position)
    print(f"  {c.cell_id}: pos={pos} az={c.azimuth_deg:.1f}deg "
          f"P={c.tx_power_dbm}dBm f={c.carrier_freq_hz/1e9:.2f}GHz "
          f"BW={c.bandwidth_hz/1e6:.0f}MHz")

print("\nFirst 3 UEs:")
for u in network.ues[:3]:
    pos = tuple(round(p, 1) for p in u.position)
    print(f"  {u.ue_id}: pos={pos} demand={u.traffic_demand_mbps:.1f}Mbps "
          f"NF={u.noise_figure_db}dB")

In [ ]:
# Top-down map: building footprints + base stations (with sector azimuths) + UEs.
def plot_footprints(ax):
    for p in bldg_files:
        v = read_ply(p)
        ring = v[: len(v) // 2, :2]            # bottom ring (z = 0)
        ring = np.vstack([ring, ring[0]])      # close the polygon
        ax.fill(ring[:, 0], ring[:, 1], facecolor="0.85",
                edgecolor="0.5", lw=0.5, zorder=1)


fig, ax = plt.subplots(figsize=(9, 9))
plot_footprints(ax)

ax.scatter([u.position[0] for u in network.ues],
           [u.position[1] for u in network.ues],
           c="tab:blue", s=25, label="UE", zorder=3)

for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=90, zorder=4)
    ax.arrow(x, y, 25 * math.cos(a), 25 * math.sin(a),
             head_width=6, color="red", zorder=4)
ax.scatter([], [], c="red", marker="^", s=90, label="cell (BS sector)")

ax.set_aspect("equal")
ax.set_xlabel("x (m, East)")
ax.set_ylabel("y (m, North)")
ax.set_title("Top-down: buildings, base stations (sectors), and UEs")
ax.legend(loc="upper right")
plt.show()

## 4. Stage 3 — Sionna RT propagation (path gain)

We use the codebase's `SionnaPropagationEngine`, which places one transmitter per
cell and one receiver per UE, runs the `PathSolver`, and reduces the result to a
per-link **path-gain matrix** of shape `(num_ues, num_cells)` in dB. This is the
exact quantity fed into the SINR/throughput stage.

> On CPU this is the slow step. Keep the scene small (a few cells, tens of UEs).

In [ ]:
from dtrapp.propagation import SionnaPropagationEngine

engine = SionnaPropagationEngine(os.path.join(SCENE_DIR, "scene.xml"), config)
path_gain_db = engine.compute_path_gain(network)   # (num_ues, num_cells), dB

print("path_gain_db shape:", path_gain_db.shape)
print("range: %.1f .. %.1f dB" % (path_gain_db.min(), path_gain_db.max()))

In [ ]:
# Heatmap of the per-link path gain.
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(path_gain_db, aspect="auto", cmap="viridis")
ax.set_xlabel("cell index")
ax.set_ylabel("UE index")
ax.set_xticks(range(len(network.cells)))
ax.set_xticklabels([c.cell_id for c in network.cells], rotation=90, fontsize=7)
ax.set_title("Per-link path gain (dB)")
fig.colorbar(im, label="path gain (dB)")
plt.show()

In [ ]:
# Visualize the actual ray paths. The engine leaves its transmitters/receivers
# in place after computing the gain, so we reuse its prepared scene and re-trace
# the paths purely for display.
from sionna.rt import PathSolver

viz_scene = engine._scene
paths = PathSolver()(viz_scene, max_depth=config.max_depth)

if no_preview:
    viz_scene.render_to_file(camera=aerial_cam, paths=paths,
                             filename="scene_paths.png", resolution=[900, 600])
    display(Image("scene_paths.png"))
else:
    viz_scene.preview(paths=paths)

# Labeled top-down map: every transmitter (cell) and receiver (UE) annotated
# with its ID, so you can read off which device is which and cross-reference the
# IDs against the throughput table / ue_throughput.csv below.
fig, ax = plt.subplots(figsize=(11, 11))
plot_footprints(ax)

# Receivers (UEs): blue dots + ue_id labels.
for u in network.ues:
    ax.scatter([u.position[0]], [u.position[1]], c="tab:blue", s=30, zorder=3)
    ax.annotate(u.ue_id, (u.position[0], u.position[1]),
                textcoords="offset points", xytext=(3, 3),
                fontsize=7, color="tab:blue", zorder=5)

# Transmitters (cells): red triangles + azimuth arrows; the cell_id is placed at
# the arrow tip so the 3 sectors sharing one site spread out and stay readable.
for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=110, zorder=4)
    ax.annotate("", xy=(x + 25 * math.cos(a), y + 25 * math.sin(a)), xytext=(x, y),
                arrowprops=dict(arrowstyle="->", color="red"), zorder=4)
    ax.annotate(c.cell_id, (x + 33 * math.cos(a), y + 33 * math.sin(a)),
                fontsize=8, fontweight="bold", color="darkred",
                ha="center", va="center", zorder=6)

ax.set_aspect("equal")
ax.set_xlabel("x (m, East)")
ax.set_ylabel("y (m, North)")
ax.set_title("Transmitter (cell) and receiver (UE) IDs\n"
             "match these IDs to the per-UE throughput table below")
plt.show()

## 5. Stages 4-5 — SINR & throughput

`compute_kpis` performs the full chain: received power = Ptx + path gain, each UE
attaches to its strongest cell, multi-cell SINR (signal / (interference + noise)),
then Shannon throughput with equal per-cell bandwidth sharing. The result holds
one record per UE and one per cell.

In [ ]:
from dtrapp.kpi import compute_kpis

result = compute_kpis(network, path_gain_db, config)

try:
    import pandas as pd
    ue_df = pd.DataFrame(result.ue_rows())
    cell_df = pd.DataFrame(result.cell_rows())
    print("Per-UE KPIs (first 10 rows):")
    display(ue_df.head(10))
    print("Per-cell KPIs:")
    display(cell_df)
    print("Mean UE throughput: %.2f Mbps" % ue_df["throughput_mbps"].mean())
except ImportError:
    ue_df = cell_df = None
    for u in result.ues[:10]:
        print(u)

In [ ]:
# Distributions of SINR and throughput across UEs.
sinr = np.array([u.sinr_db for u in result.ues])
tput = np.array([u.throughput_mbps for u in result.ues])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sinr, bins=15, color="tab:orange", edgecolor="k")
axes[0].set_xlabel("SINR (dB)"); axes[0].set_ylabel("# UEs")
axes[0].set_title("SINR distribution")
axes[1].hist(tput, bins=15, color="tab:green", edgecolor="k")
axes[1].set_xlabel("throughput (Mbps)"); axes[1].set_ylabel("# UEs")
axes[1].set_title("UE throughput distribution")
plt.tight_layout(); plt.show()

In [ ]:
# Map: each UE colored by its throughput, labeled with its id + Mbps, and linked
# to its serving cell (also labeled). This is the plot to use for verification:
# every device id sits next to its position and throughput.
cell_pos = {c.cell_id: c.position for c in network.cells}

fig, ax = plt.subplots(figsize=(11, 11))
plot_footprints(ax)
for u in result.ues:
    cp = cell_pos[u.serving_cell]
    ax.plot([u.x, cp[0]], [u.y, cp[1]], color="0.7", lw=0.5, zorder=2)
sc = ax.scatter([u.x for u in result.ues], [u.y for u in result.ues],
                c=tput, cmap="viridis", s=45, zorder=3)
for u in result.ues:
    ax.annotate(f"{u.ue_id} ({u.throughput_mbps:.1f})", (u.x, u.y),
                textcoords="offset points", xytext=(3, 3),
                fontsize=6, color="0.2", zorder=5)

for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=90, zorder=4)
    ax.annotate(c.cell_id, (x + 30 * math.cos(a), y + 30 * math.sin(a)),
                fontsize=8, fontweight="bold", color="darkred",
                ha="center", va="center", zorder=6)

ax.set_aspect("equal")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("UEs colored by throughput (label = ue_id and Mbps), "
             "linked to serving cell")
fig.colorbar(sc, label="throughput (Mbps)")
plt.show()

In [ ]:
# Per-cell view: attached UEs and aggregate throughput.
ids = [c.cell_id for c in result.cells]
att = [c.num_attached for c in result.cells]
cmb = [c.throughput_mbps for c in result.cells]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ids, att, color="tab:purple")
axes[0].set_title("UEs attached per cell"); axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(ids, cmb, color="tab:green")
axes[1].set_title("Aggregate throughput per cell (Mbps)")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

## 6. (Bonus) Coverage radio map

A `RadioMapSolver` sweeps a horizontal plane and computes path gain / SINR
everywhere — a great way to *see* coverage. Note this is Sionna's own grid map
(an illustrative continuous coverage view); the authoritative per-UE numbers are
the ones from `compute_kpis` above. This cell can be slow/memory-heavy on CPU —
lower `samples_per_tx` or raise `cell_size` if needed.

In [ ]:
from sionna.rt import RadioMapSolver

# Match the SINR map's noise to the config (illustrative only).
try:
    viz_scene.bandwidth = float(config.bandwidth_hz)
    viz_scene.temperature = float(config.temperature_k)
except Exception as e:
    print("note: could not set bandwidth/temperature on scene:", e)

rm = RadioMapSolver()(
    viz_scene,
    max_depth=config.max_depth,
    cell_size=(5.0, 5.0),
    center=[cx, cy, config.ue_height_m],
    size=[extent_m[2] - extent_m[0] + 40.0, extent_m[3] - extent_m[1] + 40.0],
    orientation=[0.0, 0.0, 0.0],
    samples_per_tx=10 ** 6,
)

rm.show(metric="path_gain"); plt.show()
rm.show(metric="sinr"); plt.show()

In [ ]:
# Overlay the SINR coverage map on the 3D scene.
if no_preview:
    viz_scene.render_to_file(camera=aerial_cam, radio_map=rm, rm_metric="sinr",
                             filename="radio_map_sinr.png", resolution=[900, 600])
    display(Image("radio_map_sinr.png"))
else:
    viz_scene.preview(radio_map=rm, rm_metric="sinr")

## 7. Stage 6 — write the output files

Finally we write the same artifacts the CLI produces: `ue_throughput.csv`,
`cell_throughput.csv`, and `throughput.json` in the config's `output_dir`.

In [ ]:
from dtrapp.runner.output import write_outputs

written = write_outputs(result, config)
for p in written:
    print("wrote", p)

# Peek at the per-UE CSV.
try:
    import pandas as pd
    display(pd.read_csv(os.path.join(config.output_dir, "ue_throughput.csv")).head())
except Exception as e:
    print("(install pandas to preview the CSV)", e)

## Notes & how to tweak

- **Exactness.** Every number comes from the codebase's own functions
  (`RandomNetworkSource`, `SionnaPropagationEngine`, `compute_kpis`,
  `write_outputs`), so results match a CLI run. The only reconstructed value is
  `extent_m`, recovered from the mesh vertices — exact to sub-micron (PLY files
  store coordinates to 6 decimals), which is far below ray-tracing resolution.
- **Reproducing a specific scene.** Point `CONFIG_PATH` at the *same* YAML used to
  build the scene (the `seed` must match) and `SCENE_DIR` at its `output/scene`.
- **Generating a fresh scene from the notebook** (needs internet for OSM):
  ```python
  from dtrapp.geometry import build_scene
  artifacts = build_scene(config.bbox, os.path.join(REPO_ROOT, "output", "scene"), config)
  extent_m = artifacts.extent_m   # exact extent, no reconstruction needed
  ```
- **Speed (CPU).** The path-solve and the radio map are the heavy steps. Shrink
  the bbox, reduce `num_ues`, lower `max_depth`, raise `cell_size`, or reduce
  `samples_per_tx`.
- **Real network data.** Replace the `RandomNetworkSource(...)` line with your own
  `NetworkDataSource.generate()` that emits `Cell`/`UE` objects in the same local
  metre frame as the buildings. Everything else in this notebook is unchanged.